# Construcción del dataset integrado

**Objetivo:** generar `dataset_integrado.csv` — tabla maestra clima + producción por `(región, cultivo, año, mes)` lista para **clustering** y modelado.

| Campo | Valor |
|-------|-------|
| Input MIDAGRI | `OUTPUTS/midagri_largo.csv` (generado por `01_midagri_pipeline.ipynb`) |
| Input NASA | `OUTPUTS/nasa_2020_2025.csv` (generado por `02_nasa_pipeline.ipynb`) |
| Input mapping | `BDS/mapping/mapping_cultivo_distrito_v2_pipeline.csv` (v2 canónico) |
| Output maestro | `OUTPUTS/dataset_integrado.csv` |

**Correcciones aplicadas (vs notebook 03 original):**
- Pareto-80: incluye cultivos hasta alcanzar **≥ 80%** de producción regional
- Agregado regional: `sum(min_count=1)` — meses sin dato MIDAGRI quedan como NaN, no como 0

**Decisiones:** NaN explícitos en meses faltantes; ceros estructurales conservados; sin remoción de outliers.

**Nombres de columnas:** variables NASA renombradas a español (`temp_promedio`, `precipitacion`, etc.). Ver `DOCUMENTACIÓN/dataset_integrado.md`.

### Qué hace esta celda

Setup: rutas, imports y constantes del merge MIDAGRI + NASA + mapping v2.

In [1]:
# ====================================================================
# CELDA: Setup — rutas e imports
# ====================================================================

# §0 Setup
# --- Importaciones ---
import sys
from pathlib import Path
import pandas as pd

# --- Rutas del proyecto ---
ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "SCRIPTS":
    ROOT = ROOT.parent
elif not (ROOT / "OUTPUTS").exists() and (ROOT.parent / "OUTPUTS").exists():
    ROOT = ROOT.parent

# --- Rutas del proyecto ---
RUTA_OUTPUT = ROOT / "OUTPUTS"
RUTA_OUTPUT.mkdir(parents=True, exist_ok=True)
RUTA_FIGURES = RUTA_OUTPUT / "figures"
RUTA_FIGURES.mkdir(parents=True, exist_ok=True)
RUTA_MAPPING = ROOT / "BDS" / "mapping" / "mapping_cultivo_distrito_v2_pipeline.csv"
DATASET_INTEGRADO = RUTA_OUTPUT / "dataset_integrado.csv"
DATASET_REGIONAL = RUTA_OUTPUT / "dataset_regional.csv"

pd.set_option("display.max_columns", 25)
# Salida informativa en consola
print(f"ROOT: {ROOT}")
print(f"OUTPUT: {RUTA_OUTPUT}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\USER\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\USER\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\USER\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\USER\anaconda3\Lib\site-packages

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\USER\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\USER\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\USER\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\USER\anaconda3\Lib\site-packages

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ROOT: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF
OUTPUT: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\OUTPUTS


## §1 Verificar insumos

Ejecuta primero los notebooks `01_midagri_pipeline.ipynb` y `02_nasa_pipeline.ipynb`,  
o coloca los CSV intermedios en `OUTPUTS/`.

### Qué hace esta celda

Verifica que existan `midagri_largo.csv`, `nasa_2020_2025.csv` y el mapping v2 antes de integrar.

In [2]:
# ====================================================================
# CELDA: Verificar insumos (01, 02, mapping)
# ====================================================================

# Diccionario de archivos requeridos
insumos = {
    "midagri_largo.csv": RUTA_OUTPUT / "midagri_largo.csv",
    "nasa_2020_2025.csv": RUTA_OUTPUT / "nasa_2020_2025.csv",
    "mapping": RUTA_MAPPING,
}

for nombre, ruta in insumos.items():
    ok = "OK" if ruta.exists() else "FALTA"
# Salida informativa en consola
    print(f"  [{ok}] {nombre}: {ruta}")

faltantes = [n for n, r in insumos.items() if not r.exists()]
if faltantes:
    raise FileNotFoundError(
        f"Faltan insumos: {faltantes}. Ejecuta notebooks 01 y 02 primero."
    )
print("\nInsumos listos.")


  [OK] midagri_largo.csv: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\OUTPUTS\midagri_largo.csv
  [OK] nasa_2020_2025.csv: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\OUTPUTS\nasa_2020_2025.csv
  [OK] mapping: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\BDS\mapping\mapping_cultivo_distrito_v2_pipeline.csv

Insumos listos.


## §2 Vista previa de insumos

### Qué hace esta celda

Vista previa de dimensiones y columnas de los tres insumos.

In [3]:
# ====================================================================
# CELDA: Vista previa MIDAGRI, NASA y mapping
# ====================================================================

df_midagri = pd.read_csv(insumos["midagri_largo.csv"])
df_nasa = pd.read_csv(insumos["nasa_2020_2025.csv"])
df_mapping = pd.read_csv(insumos["mapping"])

# Salida informativa en consola
print(f"MIDAGRI:  {df_midagri.shape}")
print(f"NASA:     {df_nasa.shape}")
print(f"Mapping:  {df_mapping.shape}")
print(f"\nRegiones: {sorted(df_midagri['region'].unique())}")
df_midagri.head(3)


MIDAGRI:  (23328, 6)
NASA:     (1008, 19)
Mapping:  (210, 4)

Regiones: ['Ica', 'Junin', 'La Libertad', 'Piura', 'Puno', 'San Martin']


,region,anio,mes_num,mes_nombre,cultivo,produccion_mensual
0,Ica,2020,1,Enero,aceituna,0.0
1,Ica,2020,1,Enero,aji,635.0
2,Ica,2020,1,Enero,ajo,12.0


## §3 Integración y exportación

Toda la lógica de merge, Pareto-80 y validación está **en este notebook** (sin módulos `.py` externos).

### Qué hace esta celda

Funciones de integración inline: Pareto-80 por región, merge por distrito, agregados regional y por cultivo, exportación.

In [4]:
# ====================================================================
# CELDA: Merge, Pareto-80, agregados y exportación
# ====================================================================


# --- Funciones de integración MIDAGRI + NASA + mapping (inline, trazabilidad) ---
# Constantes del análisis
COLS_CLIMA_NASA = [
    "t2m", "t2m_max", "t2m_min", "prectotcorr", "rh2m",
    "allsky_sfc_sw_dwn", "ws2m", "ps", "gwetroot", "ts", "t2mdew", "qv2m",
]
RENAME_CLIMA = {
    "t2m": "temp_promedio", "t2m_max": "temp_maxima", "t2m_min": "temp_minima",
    "prectotcorr": "precipitacion", "rh2m": "humedad_relativa",
    "allsky_sfc_sw_dwn": "radiacion_solar", "ws2m": "velocidad_viento",
    "ps": "presion_atmosferica", "gwetroot": "humedad_suelo", "ts": "temp_superficie",
    "t2mdew": "punto_rocio", "qv2m": "humedad_especifica",
}
RENAME_COLUMNAS = {
    **RENAME_CLIMA, "piso_asignado": "piso_ecologico", "piso": "piso_ecologico",
    "mes_num": "numero_mes", "mes_nombre": "mes", "produccion_mensual": "produccion_ton",
    "produccion_total_piso": "produccion_piso_ton", "n_cultivos": "num_cultivos",
}
COLS_META_B = [
    "region", "piso_asignado", "distrito", "cultivo",
    "anio", "mes_num", "mes_nombre", "produccion_mensual",
]


# Renombra columnas NASA/MIDAGRI a español
def renombrar_columnas(df):
    cols = {k: v for k, v in RENAME_COLUMNAS.items() if k in df.columns}
    return df.rename(columns=cols)


# Selecciona cultivos hasta ≥80% producción regional acumulada
def filtrar_pareto(df_b, umbral=0.80):
    cultivos_a_conservar, reporte = [], []
    for region in sorted(df_b["region"].unique()):
        prod = (
            df_b.loc[df_b["region"] == region]
            .groupby("cultivo")["produccion_mensual"]
            .sum()
            .sort_values(ascending=False)
        )
        prod = prod[prod > 0]
        if prod.empty:
            continue
        acum = prod.cumsum() / prod.sum()
        idx = acum[acum >= umbral].index[0] if (acum >= umbral).any() else acum.index[-1]
        cultivos_top = acum.loc[:idx].index.tolist()
        for c in cultivos_top:
            cultivos_a_conservar.append((region, c))
        reporte.append(f"{region:<13}: {len(cultivos_top)} cultivos ({100 * acum.loc[idx]:.1f}% acumulado)")
    cultivos_set = set(cultivos_a_conservar)
    mask = df_b.apply(lambda r: (r["region"], r["cultivo"]) in cultivos_set, axis=1)
    return df_b.loc[mask].copy(), cultivos_a_conservar, reporte


# Merge MIDAGRI+mapping+NASA; genera 3 datasets
def construir_datasets(df_midagri, df_nasa, df_mapping, umbral_pareto=0.80):
    mapping = df_mapping[["region", "cultivo", "piso_asignado", "distrito"]].copy()
    df_b = df_midagri.merge(mapping, on=["region", "cultivo"], how="left").dropna(subset=["distrito"]).copy()
    df_nasa_m = df_nasa.rename(columns={"region": "region_nasa"})
    df_b = df_b.merge(df_nasa_m, on=["distrito", "anio", "mes_num"], how="left")
    cols_drop = [c for c in ["region_nasa", "lat", "lon", "piso"] if c in df_b.columns]
    df_b = df_b.drop(columns=cols_drop)
    cols_clima = [c for c in df_b.columns if c not in COLS_META_B]
    df_b = df_b[COLS_META_B + cols_clima]
    df_a_prod = (
        df_b.groupby(["region", "piso_asignado", "distrito", "anio", "mes_num", "mes_nombre"])
        .agg(
            produccion_total_piso=("produccion_mensual", lambda x: x.sum(min_count=1)),
            n_cultivos=("cultivo", "nunique"),
        )
        .reset_index()
    )
    df_clima = df_b[["distrito", "anio", "mes_num"] + cols_clima].drop_duplicates(["distrito", "anio", "mes_num"])
    df_a = df_a_prod.merge(df_clima, on=["distrito", "anio", "mes_num"], how="left")
    df_a = df_a.rename(columns={"piso_asignado": "piso"})
    cols_meta_a = [
        "region", "piso", "distrito", "anio", "mes_num", "mes_nombre",
        "n_cultivos", "produccion_total_piso",
    ]
    df_a = df_a[cols_meta_a + cols_clima]
    df_integrado, _, _ = filtrar_pareto(df_b, umbral=umbral_pareto)
    return {"dataset_por_cultivo": df_b, "dataset_regional": df_a, "dataset_integrado": df_integrado}


# Comprueba coherencia agregado A vs suma B
def validar(dfs):
    df_b, df_a = dfs["dataset_por_cultivo"], dfs["dataset_regional"]
    suma_b = (
        df_b.groupby(["region", "piso_asignado", "anio", "mes_num"])["produccion_mensual"]
        .sum(min_count=1)
        .reset_index()
        .rename(columns={"piso_asignado": "piso", "produccion_mensual": "suma_desde_B"})
    )
    check = df_a[["region", "piso", "anio", "mes_num", "produccion_total_piso"]].merge(
        suma_b, on=["region", "piso", "anio", "mes_num"], how="outer"
    )
    check["diferencia"] = (check["produccion_total_piso"] - check["suma_desde_B"]).abs()
    max_diff = check["diferencia"].max()
    cols_clima_nasa = [c for c in COLS_CLIMA_NASA if c in df_b.columns]
    return {
        "max_diff_a_b": float(max_diff) if pd.notna(max_diff) else 0.0,
        "nan_clima": int(df_b[cols_clima_nasa].isna().sum().sum()),
        "nan_prod_b": int(df_b["produccion_mensual"].isna().sum()),
        "nan_prod_a": int(df_a["produccion_total_piso"].isna().sum()),
        "shapes": {k: v.shape for k, v in dfs.items()},
    }


# Escribe CSVs en OUTPUTS/
def exportar(dfs, ruta_output):
    ruta_output.mkdir(parents=True, exist_ok=True)
    for name, df in dfs.items():
        out = renombrar_columnas(df.copy())
        if "produccion_ton" in out.columns:
            out["produccion_ton"] = out["produccion_ton"].round(4)
        if "produccion_piso_ton" in out.columns:
            out["produccion_piso_ton"] = out["produccion_piso_ton"].round(4)
        out.to_csv(ruta_output / f"{name}.csv", index=False, encoding="utf-8-sig")
    integrado = ruta_output / "dataset_integrado.csv"
    filtrado = ruta_output / "dataset_por_cultivo_filtrado.csv"
    if integrado.exists():
        filtrado.write_bytes(integrado.read_bytes())

# Umbral Pareto-80 por región
UMBRAL_PARETO = 0.80
# Ejecutar pipeline de integración
dfs = construir_datasets(df_midagri, df_nasa, df_mapping, umbral_pareto=UMBRAL_PARETO)
stats = validar(dfs)
_, _, reporte_pareto = filtrar_pareto(dfs['dataset_por_cultivo'], umbral=UMBRAL_PARETO)
exportar(dfs, RUTA_OUTPUT)
df_integrado = renombrar_columnas(dfs['dataset_integrado'])

# Salida informativa en consola
print("=== Pareto-80 (corregido) ===")
for line in reporte_pareto:
    print(line)

# Salida informativa en consola
print("\n=== Validación ===")
for k, v in stats.items():
    print(f'  {k}: {v}')

# Salida informativa en consola
print(f"\nDataset integrado: {df_integrado.shape}")
df_integrado.head()


=== Pareto-80 (corregido) ===
Ica          : 8 cultivos (80.7% acumulado)
Junin        : 9 cultivos (82.2% acumulado)
La Libertad  : 4 cultivos (81.8% acumulado)
Piura        : 5 cultivos (82.8% acumulado)
Puno         : 3 cultivos (89.9% acumulado)
San Martin   : 4 cultivos (83.8% acumulado)

=== Validación ===
  max_diff_a_b: 4.656612873077393e-10
  nan_clima: 0
  nan_prod_b: 1082
  nan_prod_a: 56
  shapes: {'dataset_por_cultivo': (15120, 20), 'dataset_regional': (1008, 20), 'dataset_integrado': (2376, 20)}

Dataset integrado: (2376, 20)


,region,piso_ecologico,distrito,cultivo,anio,numero_mes,mes,produccion_ton,temp_promedio,temp_maxima,temp_minima,precipitacion,humedad_relativa,radiacion_solar,velocidad_viento,presion_atmosferica,humedad_suelo,temp_superficie,punto_rocio,humedad_especifica
4,Ica,costa,Chincha Alta,alfalfa,2020,1,Enero,10684.89,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
11,Ica,costa,Chincha Alta,esparrago,2020,1,Enero,14799.60,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
15,Ica,costa,Chincha Alta,maiz_amarillo_duro,2020,1,Enero,9000.19,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
18,Ica,costa,Chincha Alta,mandarina,2020,1,Enero,0.00,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
22,Ica,costa,Chincha Alta,palta,2020,1,Enero,7.10,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41


### Qué produce esta celda

Genera tres CSV en `OUTPUTS/`:
- `dataset_integrado.csv` — tabla maestra (cultivo×mes)
- `dataset_regional.csv` — Análisis A (piso×mes)
- `dataset_por_cultivo.csv` — series por cultivo Pareto

Aplica `sum(min_count=1)` en agregados para no convertir meses 100% NaN en cero.

## §4 Resumen del dataset maestro

`dataset_integrado.csv` = cultivos Pareto-80 × 72 meses × clima asignado por piso ecológico.

**Uso en clustering:**
```python
df = pd.read_csv("OUTPUTS/dataset_integrado.csv")
```

### Qué hace esta celda

Lee `dataset_integrado.csv` generado y resume filas, combos Pareto y NaN intencionales.

In [5]:
# ====================================================================
# CELDA: Resumen del dataset_integrado.csv exportado
# ====================================================================

df_integrado = pd.read_csv(RUTA_OUTPUT / "dataset_integrado.csv")

# Salida informativa en consola
print("=== dataset_integrado.csv ===")
print(f"Filas: {len(df_integrado):,}  |  Columnas: {len(df_integrado.columns)}")
print(f"Combinaciones (region, cultivo): {df_integrado.groupby(['region','cultivo']).ngroups}")
print(f"NaN en produccion_ton: {df_integrado['produccion_ton'].isna().sum()}")
print(f"\nColumnas:\n  {list(df_integrado.columns)}")
print("\n=== Archivos exportados ===")
for f in sorted(RUTA_OUTPUT.glob("dataset*.csv")):
    n = sum(1 for _ in open(f, encoding='utf-8-sig')) - 1
    print(f"  {f.name}: {n:,} filas")

df_integrado.describe().round(2)


=== dataset_integrado.csv ===
Filas: 2,376  |  Columnas: 20
Combinaciones (region, cultivo): 33
NaN en produccion_ton: 166

Columnas:
  ['region', 'piso_ecologico', 'distrito', 'cultivo', 'anio', 'numero_mes', 'mes', 'produccion_ton', 'temp_promedio', 'temp_maxima', 'temp_minima', 'precipitacion', 'humedad_relativa', 'radiacion_solar', 'velocidad_viento', 'presion_atmosferica', 'humedad_suelo', 'temp_superficie', 'punto_rocio', 'humedad_especifica']

=== Archivos exportados ===
  dataset_integrado.csv: 2,376 filas
  dataset_por_cultivo.csv: 15,120 filas
  dataset_por_cultivo_filtrado.csv: 2,376 filas
  dataset_regional.csv: 1,008 filas


,anio,numero_mes,produccion_ton,temp_promedio,temp_maxima,temp_minima,precipitacion,humedad_relativa,radiacion_solar,velocidad_viento,presion_atmosferica,humedad_suelo,temp_superficie,punto_rocio,humedad_especifica
count,2376.00,2376.00,2210.00,2376.00,2376.00,2376.00,2376.00,2376.00,2376.00,2376.00,2376.00,2376.00,2376.00,2376.00,2376.00
mean,2022.50,6.50,47918.67,18.66,26.87,12.50,0.89,70.56,18.48,2.20,86.10,0.45,20.07,12.45,11.00
std,1.71,3.45,130974.60,6.09,5.87,7.30,1.33,7.87,3.38,1.35,13.30,0.10,5.88,6.08,2.64
min,2020.00,1.00,0.00,3.34,13.87,-10.38,0.00,36.46,10.56,0.01,60.92,0.32,3.86,-8.94,3.42
25%,2021.00,3.75,3500.00,17.01,23.40,11.53,0.08,64.27,15.79,0.82,82.49,0.38,17.81,10.41,8.88
50%,2022.50,6.50,16156.35,20.05,27.01,15.09,0.29,71.23,18.49,2.35,90.47,0.42,21.15,14.54,11.19
75%,2024.00,9.25,37506.25,23.00,31.14,17.37,1.16,76.91,21.03,3.19,95.89,0.52,23.85,16.86,13.03
max,2025.00,12.00,3046061.42,29.19,38.72,23.43,9.15,90.47,29.31,4.86,99.69,0.76,31.46,22.37,17.12
